In [29]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt',
    'names.txt'
)

('names.txt', <http.client.HTTPMessage at 0x2e416f16190>)

In [30]:
words = open('names.txt', 'r').read().splitlines()

In [31]:
import torch
import math
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [50]:
# string to integer mapping and reverse mapping
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [51]:
# randomly shuffling the words and initialising the sizes of splits
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

In [52]:
#building the train, test and validation splits
block_size = 3
def build_set(words):
    X = []
    Y = []
    for w in words:
        context = [0]*block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

Xtr, Ytr = build_set(words[:n1])
Xdev, Ydev = build_set(words[n1:n2])
Xtest, Ytest = build_set(words[n2:])

In [53]:
# initialising all the parameters of the neural network
g = torch.Generator().manual_seed(42)
embedding_size = 10
n_hidden = 200
C = torch.randn((27, embedding_size), generator = g)
W1 = torch.randn((block_size * embedding_size, n_hidden), generator = g)
b1 = torch.randn(n_hidden, generator = g)
W2 = torch.randn((n_hidden, 27), generator = g)
b2 = torch.randn(27, generator = g)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True

In [54]:
max_steps = 200000
# lre = torch.linspace(-3, 0, max_steps)
# lrs = 10**lre
lri = []
lossi = []
# training the neural network by splitting into random batches.

In [55]:
batch_size = 32
for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator = g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    emb = C[Xb]
    embcat = emb.view(-1, embedding_size * block_size)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    for p in parameters:
        p.grad = None

    loss.backward()

    lossi.append(loss.item())
    lr = 0.1 if i <= 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad


In [ ]:
vocab_size = 27

In [56]:
# finding loss of the train and dev splits.
@torch.no_grad # decorator that no gradients are required for the find_loss function
def find_loss(split):
    X, Y = {
        'train': (Xtr, Ytr),
        'dev': (Xdev, Ydev),
        'test': (Xtest, Ytest)
    }[split]
    emb = C[X]
    h = torch.tanh(emb.view(-1, block_size * embedding_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(split, loss.item())

find_loss('train')
find_loss('dev')

train 2.1141462326049805
dev 2.1557254791259766


In [57]:
find_loss('test')
g = torch.Generator().manual_seed(42)
for i in range(20):
    out = []
    context = [0]*block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(-1, block_size * embedding_size) @ W1 + b1)
        logits = h @ W2 + b2
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims = True)
        ix = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g)
        out.append(ix.item())
        if ix == 0:
            break
        context = context[1:] + [ix]
    print(''.join(itos[i] for i in out))

test 2.1687285900115967
anuelen.
tis.
mari.
nedyn.
shan.
silaylen.
kemarce.
man.
epiannalei.
dazi.
kence.
jordon.
kalla.
mikhlynn.
acvinia.
ana.
lavittareth.
jaymar.
tal.
selix.
